In [5]:
#!/usr/bin/env Rscript

# =============================================================================
# MDS_index_profiles_OHFP_compute_v2.R
# -----------------------------------------------------------------------------
# Purpose:
#   Compute ONLY:
#     - classical MDS (cmdscale)  &  non-metric MDS (isoMDS)
#     - For each:
#         method_tag ∈ {classical, nonmetric}
#         U_tag      ∈ {U10, U25, U50}  (max k)
#         dataset    ∈ {OH, FP}
#         mode       ∈ {top3, cumeig}
#       → Ward.D2 (Euclidean) clustering for k = 2..K_max
#       → indices: silhouette, dunn, gap, ch, db, ptbiserial
#
# Inputs:
#   <base_root>/20251026_under_25clusters_program_and_result/for_checking_20251127/data/for_MDS_STEP2/
#     - preprocessed_features_OH.csv
#     - preprocessed_features_FP.csv
#
# Outputs (CSV only; NO figures here):
#   <base_root>/20251026_under_25clusters_program_and_result/for_checking_20251127/4.2_clustering/OH_FP_only/
#     {method_tag}/{U_tag}/{dataset}/csv_profiles/
#       - index_profile_{mode}_{index}_{method_tag}_{U_tag}_kmax{K}_{TS}.csv
#       - index_profiles_all_{dataset}_{mode}_{method_tag}_{U_tag}_kmax{K}_{TS}.csv
#
#   (Script 2 for plotting / summaries can reuse these CSVs.)
# =============================================================================

# ==== CONFIG (edit here) =====================================================

base_root <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No"

# Input feature files (OH/FP) for MDS
data_dir <- file.path(
  base_root,
  "20251026_under_25clusters_program_and_result",
  "for_checking_20251127",
  "data",
  "for_MDS_STEP2"
)

# Root of all outputs for this script
out_root <- file.path(
  base_root,
  "20251026_under_25clusters_program_and_result",
  "for_checking_20251127",
  "4.2_clustering",
  "OH_FP_only"
)

# Max cluster numbers: k = 2..K_max for each tag
max_k_list <- c(U10 = 10, U25 = 25, U50 = 50)

# MDS methods
method_tags <- c("classical", "nonmetric")

# Gap statistic bootstrap iterations
gap_B <- 20

# Input feature file names
dataset_files <- c(
  OH = "preprocessed_features_OH.csv",
  FP = "preprocessed_features_FP.csv"
)

# Dimension selection modes
dim_modes   <- c("top3", "cumeig")

# Indices to compute
index_names <- c("silhouette", "dunn", "gap", "ch", "db", "ptbiserial")

# Timestamp for this compute run
ts_str <- format(Sys.time(), "%Y%m%d_%H%M%S")

# =============================================================================
# Libraries & basic helpers
# =============================================================================

safe_lib <- function(pkg){
  if (!require(pkg, character.only = TRUE)) {
    install.packages(pkg, repos = "https://cloud.r-project.org")
    library(pkg, character.only = TRUE)
  }
}
safe_lib("readr")
safe_lib("cluster")  # silhouette, clusGap
safe_lib("fpc")      # dunn via cluster.stats
safe_lib("MASS")     # isoMDS

set.seed(42)

ensure_dir <- function(p){
  if (!dir.exists(p)) dir.create(p, recursive = TRUE, showWarnings = FALSE)
}

read_numeric_matrix <- function(path){
  df <- read.delim(path, header = TRUE, sep = ",",
                   row.names = 1, as.is = TRUE, strip.white = FALSE)
  is_num <- !vapply(df, is.character, logical(1))
  if (!any(is_num)) stop("No numeric columns: ", path)
  df[, is_num, drop = FALSE]
}

drop_zero_var_cols <- function(M){
  M <- as.matrix(M)
  keep <- apply(M, 2, function(v) sd(v, na.rm=TRUE) > 0)
  if (!any(keep)) stop("All columns have zero variance.")
  M[, keep, drop=FALSE]
}

# =============================================================================
# MDS (classical) from correlation + distance
# =============================================================================

compute_mds_and_dist <- function(numData, k_max_eig = 50){
  corData <- suppressWarnings(cor(numData, use = "pairwise.complete.obs"))
  corData[is.na(corData)] <- 0
  ddata <- dist(1 - corData)

  k <- min(k_max_eig, max(1, ncol(corData) - 1))
  mds <- cmdscale(ddata, k = k, eig = TRUE)

  eig_vals <- mds$eig
  totaleig <- sum(abs(eig_vals))
  peigen   <- if (totaleig == 0) rep(0, length(eig_vals)) else abs(eig_vals) / totaleig
  contrib <- data.frame(
    axis        = seq_along(eig_vals),
    eigenvalue  = eig_vals,
    percent     = peigen,
    cum_percent = cumsum(peigen),
    stringsAsFactors = FALSE
  )
  list(D = ddata, mds = mds, contrib = contrib)
}

select_dims <- function(contrib_df, mode, avail_dims){
  pos_idx <- which(contrib_df$percent > 0)
  if (length(pos_idx) == 0) return(min(3, max(1, avail_dims)))
  if (mode == "top3"){
    k <- min(3, length(pos_idx))
  } else {
    thr <- 0.8
    k <- which(contrib_df$cum_percent[pos_idx] >= thr)[1]
    if (is.na(k)) k <- length(pos_idx)
  }
  k <- min(k, avail_dims); k <- max(1, k)
  k
}

# =============================================================================
# Index implementations (CH / DB / point-biserial)
# =============================================================================

compute_db_index <- function(X, cl){
  if (length(unique(cl)) < 2) return(NA_real_)
  X <- as.matrix(X)
  labs <- sort(unique(cl))
  k <- length(labs)

  cent <- sapply(labs, function(L){ colMeans(X[cl == L, , drop = FALSE]) })
  cent <- t(cent)

  S <- numeric(k)
  for (i in seq_len(k)){
    idx <- which(cl == labs[i])
    if (length(idx) <= 1){
      S[i] <- 0
    } else {
      ci <- cent[i, , drop = FALSE]
      S[i] <- mean(sqrt(rowSums((X[idx, , drop = FALSE] -
                                   matrix(ci, nrow=length(idx), ncol=ncol(X), byrow = TRUE))^2)))
    }
  }

  Dc <- as.matrix(dist(cent))
  R <- matrix(NA_real_, k, k)
  for (i in seq_len(k)){
    for (j in seq_len(k)){
      if (i == j) next
      if (Dc[i, j] == 0) R[i, j] <- Inf else R[i, j] <- (S[i] + S[j]) / Dc[i, j]
    }
  }
  db <- mean(apply(R, 1, function(v) max(v[is.finite(v)], na.rm = TRUE)), na.rm = TRUE)
  if (!is.finite(db)) return(NA_real_)
  db
}

compute_ptbiserial_index <- function(D, cl){
  n <- attr(D, "Size")
  if (length(unique(cl)) < 2 || n < 2) return(NA_real_)
  s <- logical(n * (n - 1) / 2)
  idx <- 1
  for (i in 1:(n-1)){
    for (j in (i+1):n){
      s[idx] <- (cl[i] == cl[j]); idx <- idx + 1
    }
  }
  d <- as.numeric(D)
  if (length(unique(s)) < 2) return(NA_real_)
  r <- suppressWarnings(cor(d, as.numeric(s), method = "pearson", use = "complete.obs"))
  if (is.na(r)) return(NA_real_)
  -r
}

compute_ch_index <- function(X, cl){
  X <- as.matrix(X)
  n <- nrow(X)
  labs <- sort(unique(cl))
  k <- length(labs)
  if (k < 2 || n <= k) return(NA_real_)

  m_all <- colMeans(X)
  ssb <- 0
  ssw <- 0
  for (g in labs){
    idx <- which(cl == g)
    Xg  <- X[idx, , drop = FALSE]
    mg  <- colMeans(Xg)
    ssb <- ssb + nrow(Xg) * sum((mg - m_all)^2)
    ssw <- ssw + sum(rowSums((Xg - matrix(mg, nrow = nrow(Xg),
                                          ncol = ncol(Xg), byrow = TRUE))^2))
  }
  if (ssw <= 0) return(NA_real_)
  (ssb / (k - 1)) / (ssw / (n - k))
}

# =============================================================================
# Get MDS coordinates (classical / nonmetric)
# =============================================================================

get_X_for <- function(dataset_tag = c("OH","FP"),
                      mode = c("top3","cumeig"),
                      method_tag = c("classical","nonmetric")){
  dataset_tag <- match.arg(dataset_tag)
  mode        <- match.arg(mode)
  method_tag  <- match.arg(method_tag)

  csv_name <- dataset_files[[dataset_tag]]
  in_csv <- file.path(data_dir, csv_name)
  if (!file.exists(in_csv)) stop("Input file not found: ", in_csv)

  message(sprintf("\n[Info] Reading features: %s (%s)", in_csv, dataset_tag))
  numData <- read_numeric_matrix(in_csv)

  md <- compute_mds_and_dist(numData)
  D   <- md$D
  mds <- md$mds
  contrib <- md$contrib

  coords_classical <- as.matrix(mds$points)
  if (is.null(dim(coords_classical))) {
    coords_classical <- matrix(coords_classical, ncol = 1)
    colnames(coords_classical) <- "Dim1"
  }

  k_dim <- select_dims(contrib, mode, ncol(coords_classical))
  message(sprintf("[Info] %s / %s: selected dim = %d (method=%s)",
                  dataset_tag, mode, k_dim, method_tag))

  if (method_tag == "classical"){
    X <- coords_classical[, 1:k_dim, drop = FALSE]
  } else {
    init <- coords_classical[, 1:k_dim, drop = FALSE]
    iso <- try(MASS::isoMDS(D, y = init, k = k_dim, maxit = 50, trace = TRUE),
               silent = TRUE)
    if (inherits(iso, "try-error")){
      message("  [Warn] isoMDS failed → fall back to classical coords")
      X <- init
    } else {
      message(sprintf("  [Info] isoMDS final stress: %.4f", iso$stress))
      X <- iso$points
    }
  }

  X <- scale(drop_zero_var_cols(X))
  list(X = X, D = dist(X))
}

# =============================================================================
# Index profile computation over k = 2..K_max
# =============================================================================

ward_cut <- function(x, k){
  cl <- cutree(hclust(dist(x), method = "ward.D2"), k)
  list(cluster = cl)
}

compute_index_profiles <- function(
  X,
  D,
  dataset_tag,
  mode_tag,
  U_tag,
  k_max,
  out_csv_dir,
  method_tag
){
  ensure_dir(out_csv_dir)

  n <- nrow(X)
  k_max <- min(k_max, max(2, n - 1))
  ks <- 2:k_max

  message(sprintf("\n=== [Compute] method=%s, dataset=%s, mode=%s, U=%s (k=2..%d) ===",
                  method_tag, dataset_tag, mode_tag, U_tag, k_max))

  # ---- GAP (cluster::clusGap) ----
  gap_vec <- rep(NA_real_, length(ks))
  if (n >= 3) {
    set.seed(42)
    gap_obj <- try(
      cluster::clusGap(X, FUNcluster = ward_cut,
                       K.max = k_max, B = gap_B, verbose = FALSE),
      silent = TRUE
    )
    if (!inherits(gap_obj, "try-error")) {
      gap_tab <- as.data.frame(gap_obj$Tab)
      gap_tab$k <- seq_len(nrow(gap_tab))
      gap_tab <- gap_tab[gap_tab$k %in% ks, c("k","gap")]
      for (j in seq_len(nrow(gap_tab))) {
        pos <- which(ks == gap_tab$k[j])
        if (length(pos) == 1) gap_vec[pos] <- gap_tab$gap[j]
      }
    } else {
      warning("clusGap failed for ", dataset_tag, "/", mode_tag, "/", U_tag,
              " (method=", method_tag, "): ",
              conditionMessage(attr(gap_obj, "condition")))
    }
  }

  # ---- other indices over k ----
  sil  <- dunn <- db <- ch <- ptb <- rep(NA_real_, length(ks))

  hc <- hclust(D, method = "ward.D2")

  for (i in seq_along(ks)){
    k <- ks[i]
    cl_raw <- cutree(hc, k)
    cl <- as.integer(factor(cl_raw))
    k_eff <- length(unique(cl))
    sizes <- sort(table(cl))

    message(sprintf("\n[DEBUG] %s/%s/%s/%s k=%d (eff=%d), sizes: %s",
                    method_tag, dataset_tag, mode_tag, U_tag,
                    k, k_eff, paste(sizes, collapse = ",")))

    if (k_eff < 2){
      message("  -> k_eff < 2, all indices set to NA")
      next
    }

    # silhouette
    sil[i] <- tryCatch({
      sil_obj <- cluster::silhouette(cl, D)
      val <- mean(sil_obj[,3], na.rm = TRUE)
      message(sprintf("  silhouette = %s",
                      ifelse(is.na(val), "NA", sprintf("%.6f", val))))
      val
    }, error = function(e){
      message(sprintf("  [ERROR silhouette] %s", conditionMessage(e)))
      NA_real_
    })

    # dunn (via fpc::cluster.stats)
    dstat <- try(fpc::cluster.stats(D, cl), silent = TRUE)
    if (!inherits(dstat, "try-error") && !is.null(dstat$dunn)){
      dunn[i] <- dstat$dunn
      message(sprintf("  dunn = %s",
                      ifelse(is.na(dunn[i]), "NA", sprintf("%.6f", dunn[i]))))
    } else if (inherits(dstat, "try-error")){
      message(sprintf("  [ERROR dunn] %s",
                      conditionMessage(attr(dstat, "condition"))))
    }

    # DB
    db[i] <- tryCatch({
      val <- compute_db_index(X, cl)
      message(sprintf("  DB = %s",
                      ifelse(is.na(val), "NA", sprintf("%.6f", val))))
      val
    }, error = function(e){
      message(sprintf("  [ERROR DB] %s", conditionMessage(e)))
      NA_real_
    })

    # CH
    ch[i] <- tryCatch({
      val <- compute_ch_index(X, cl)
      message(sprintf("  CH = %s",
                      ifelse(is.na(val), "NA", sprintf("%.6f", val))))
      val
    }, error = function(e){
      message(sprintf("  [ERROR CH] %s", conditionMessage(e)))
      NA_real_
    })

    # ptbiserial
    ptb[i] <- tryCatch({
      val <- compute_ptbiserial_index(D, cl)
      message(sprintf("  ptbiserial = %s",
                      ifelse(is.na(val), "NA", sprintf("%.6f", val))))
      val
    }, error = function(e){
      message(sprintf("  [ERROR ptbiserial] %s", conditionMessage(e)))
      NA_real_
    })
  }

  # ---- save per-index CSVs ----
  profiles <- list(
    silhouette = data.frame(k = ks, score = sil),
    dunn       = data.frame(k = ks, score = dunn),
    gap        = data.frame(k = ks, score = gap_vec),
    ch         = data.frame(k = ks, score = ch),
    db         = data.frame(k = ks, score = db),
    ptbiserial = data.frame(k = ks, score = ptb)
  )

  for (ix in names(profiles)){
    df <- profiles[[ix]]
    out_f <- file.path(
      out_csv_dir,
      sprintf("index_profile_%s_%s_%s_%s_kmax%d_%s.csv",
              mode_tag, ix, method_tag, U_tag, k_max, ts_str)
    )
    readr::write_csv(df, out_f)
    message("[Saved] ", out_f)
  }

  # ---- save combined all-k CSV ----
  all_k_df <- data.frame(
    k          = ks,
    silhouette = sil,
    dunn       = dunn,
    gap        = gap_vec,
    ch         = ch,
    db         = db,
    ptbiserial = ptb,
    stringsAsFactors = FALSE
  )
  out_all <- file.path(
    out_csv_dir,
    sprintf("index_profiles_all_%s_%s_%s_%s_kmax%d_%s.csv",
            dataset_tag, mode_tag, method_tag, U_tag, k_max, ts_str)
  )
  readr::write_csv(all_k_df, out_all)
  message("[Saved all-k indices] ", out_all)

  invisible(NULL)
}

# =============================================================================
# MAIN
# =============================================================================

ensure_dir(out_root)

for (method_tag in method_tags){
  message(sprintf("\n================ METHOD = %s ================\n", method_tag))
  method_root <- file.path(out_root, method_tag)
  ensure_dir(method_root)

  for (U_tag in names(max_k_list)){
    k_max <- max_k_list[[U_tag]]
    message(sprintf("\n######## U-tag = %s (k_max = %d) ########", U_tag, k_max))

    out_U_root <- file.path(method_root, U_tag)
    ensure_dir(out_U_root)

    for (dataset_tag in names(dataset_files)){
      out_ds_root <- file.path(out_U_root, dataset_tag)
      out_ds_csv  <- file.path(out_ds_root, "csv_profiles")
      ensure_dir(out_ds_root); ensure_dir(out_ds_csv)

      for (mode_tag in dim_modes){

        mdx <- get_X_for(dataset_tag, mode_tag, method_tag)
        X <- as.matrix(mdx$X)
        D <- mdx$D

        message(sprintf("[Info] %s / %s / %s / %s: n=%d, p=%d",
                        method_tag, U_tag, dataset_tag, mode_tag,
                        nrow(X), ncol(X)))

        compute_index_profiles(
          X          = X,
          D          = D,
          dataset_tag = dataset_tag,
          mode_tag    = mode_tag,
          U_tag       = U_tag,
          k_max       = k_max,
          out_csv_dir = out_ds_csv,
          method_tag  = method_tag
        )
      }
    }
  }
}

message("\n✅ Compute finished. CSV profiles are under: ", out_root)



================ METHOD = classical ================



######## U-tag = U10 (k_max = 10) ########


[Info] Reading features: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/for_MDS_STEP2/preprocessed_features_OH.csv (OH)

[Info] OH / top3: selected dim = 3 (method=classical)

[Info] classical / U10 / OH / top3: n=394, p=3


=== [Compute] method=classical, dataset=OH, mode=top3, U=U10 (k=2..10) ===


[DEBUG] classical/OH/top3/U10 k=2 (eff=2), sizes: 20,374

  silhouette = 0.774703

  dunn = 0.109484

  DB = 1.123348

  CH = 132.625585

  ptbiserial = 0.719927


[DEBUG] classical/OH/top3/U10 k=3 (eff=3), sizes: 7,13,374

  silhouette = 0.779770

  dunn = 0.109484

  DB = 0.446201

  CH = 185.494444

  ptbiserial = 0.729326


[DEBUG] classical/OH/top3/U10 k=4 (eff=4), sizes: 7,9,13,365

  silhouette = 0.763396

  dunn = 0.063974

  DB = 0.424894

  CH = 226.244865

  ptbiserial = 0.814971


[DEBUG] class


[DEBUG] classical/FP/top3/U10 k=3 (eff=3), sizes: 18,27,206

  silhouette = 0.409074

  dunn = 0.066269

  DB = 0.687778

  CH = 81.540741

  ptbiserial = 0.518676


[DEBUG] classical/FP/top3/U10 k=4 (eff=4), sizes: 18,24,27,182

  silhouette = 0.450378

  dunn = 0.074079

  DB = 0.605341

  CH = 113.370425

  ptbiserial = 0.618795


[DEBUG] classical/FP/top3/U10 k=5 (eff=5), sizes: 18,24,27,65,117

  silhouette = 0.430583

  dunn = 0.036502

  DB = 0.906897

  CH = 137.845432

  ptbiserial = 0.618085


[DEBUG] classical/FP/top3/U10 k=6 (eff=6), sizes: 15,18,24,27,50,117

  silhouette = 0.476130

  dunn = 0.039573

  DB = 0.810055

  CH = 161.536591

  ptbiserial = 0.650550


[DEBUG] classical/FP/top3/U10 k=7 (eff=7), sizes: 15,18,21,24,27,29,117

  silhouette = 0.519001

  dunn = 0.052893

  DB = 0.609654

  CH = 207.640308

  ptbiserial = 0.670345


[DEBUG] classical/FP/top3/U10 k=8 (eff=8), sizes: 7,15,18,21,22,24,27,117

  silhouette = 0.493239

  dunn = 0.052893

  DB = 0.565307


  silhouette = 0.530797

  dunn = 0.011605

  DB = 0.566888

  CH = 243.734914

  ptbiserial = 0.629725


[DEBUG] classical/OH/top3/U25 k=7 (eff=7), sizes: 7,7,9,13,19,46,293

  silhouette = 0.569217

  dunn = 0.012315

  DB = 0.631964

  CH = 256.368570

  ptbiserial = 0.629309


[DEBUG] classical/OH/top3/U25 k=8 (eff=8), sizes: 2,7,7,9,13,19,46,291

  silhouette = 0.587297

  dunn = 0.014689

  DB = 0.593318

  CH = 277.240816

  ptbiserial = 0.652428


[DEBUG] classical/OH/top3/U25 k=9 (eff=9), sizes: 2,6,7,7,7,9,19,46,291

  silhouette = 0.596014

  dunn = 0.018807

  DB = 0.531303

  CH = 314.221233

  ptbiserial = 0.653511


[DEBUG] classical/OH/top3/U25 k=10 (eff=10), sizes: 2,6,7,7,7,9,11,19,46,280

  silhouette = 0.624337

  dunn = 0.018807

  DB = 0.560772

  CH = 364.070396

  ptbiserial = 0.667017


[DEBUG] classical/OH/top3/U25 k=11 (eff=11), sizes: 2,6,7,7,7,9,10,11,19,36,280

  silhouette = 0.577899

  dunn = 0.018807

  DB = 0.565149

  CH = 382.141237

  ptbiserial = 0

  silhouette = 0.268405

  dunn = 0.293546

  DB = 1.126263

  CH = 8.649532

  ptbiserial = 0.462546


[DEBUG] classical/OH/cumeig/U25 k=15 (eff=15), sizes: 2,2,2,2,2,3,3,4,4,4,5,5,12,25,319

  silhouette = 0.268799

  dunn = 0.199863

  DB = 1.357817

  CH = 8.773663

  ptbiserial = 0.478971


[DEBUG] classical/OH/cumeig/U25 k=16 (eff=16), sizes: 2,2,2,2,2,2,3,3,4,4,4,5,5,10,25,319

  silhouette = 0.275950

  dunn = 0.199863

  DB = 1.291878

  CH = 8.903077

  ptbiserial = 0.479747


[DEBUG] classical/OH/cumeig/U25 k=17 (eff=17), sizes: 2,2,2,2,2,2,3,3,4,4,4,5,5,10,17,25,302

  silhouette = 0.255303

  dunn = 0.168493

  DB = 1.361372

  CH = 9.042706

  ptbiserial = 0.483090


[DEBUG] classical/OH/cumeig/U25 k=18 (eff=18), sizes: 2,2,2,2,2,2,3,3,4,4,4,5,5,10,13,17,25,289

  silhouette = 0.222175

  dunn = 0.168493

  DB = 1.398811

  CH = 9.183527

  ptbiserial = 0.474622


[DEBUG] classical/OH/cumeig/U25 k=19 (eff=19), sizes: 2,2,2,2,2,2,3,3,3,4,4,4,5,5,10,13,17,22,289

  silhouet

  silhouette = 0.504147

  dunn = 0.075361

  DB = 0.695256

  CH = 344.388731

  ptbiserial = 0.406539


[DEBUG] classical/FP/top3/U25 k=22 (eff=22), sizes: 2,3,3,5,5,7,7,8,8,9,11,11,11,13,13,14,14,17,17,18,24,31

  silhouette = 0.506681

  dunn = 0.076582

  DB = 0.726806

  CH = 347.172907

  ptbiserial = 0.396040


[DEBUG] classical/FP/top3/U25 k=23 (eff=23), sizes: 2,3,3,4,5,5,7,7,8,8,9,10,11,11,11,13,13,14,17,17,18,24,31

  silhouette = 0.507910

  dunn = 0.076582

  DB = 0.720313

  CH = 351.396701

  ptbiserial = 0.393351


[DEBUG] classical/FP/top3/U25 k=24 (eff=24), sizes: 2,3,3,4,5,5,5,6,7,7,8,8,9,10,11,11,13,13,14,17,17,18,24,31

  silhouette = 0.510211

  dunn = 0.076582

  DB = 0.713611

  CH = 354.150606

  ptbiserial = 0.391262


[DEBUG] classical/FP/top3/U25 k=25 (eff=25), sizes: 2,3,3,4,4,5,5,5,5,6,7,7,8,8,10,11,11,13,13,14,17,17,18,24,31

  silhouette = 0.517092

  dunn = 0.086787

  DB = 0.692798

  CH = 357.839682

  ptbiserial = 0.390074

[Saved] /Volumes/csbdeep1

...

  [Info] isoMDS final stress: 8.0672

[Info] nonmetric / U50 / FP / top3: n=251, p=3


=== [Compute] method=nonmetric, dataset=FP, mode=top3, U=U50 (k=2..50) ===


[DEBUG] nonmetric/FP/top3/U50 k=2 (eff=2), sizes: 81,170

  silhouette = 0.244586

  dunn = 0.060553

  DB = 1.553321

  CH = 71.268595

  ptbiserial = 0.298288


[DEBUG] nonmetric/FP/top3/U50 k=3 (eff=3), sizes: 24,81,146

  silhouette = 0.289686

  dunn = 0.060553

  DB = 1.214822

  CH = 76.440412

  ptbiserial = 0.396309


[DEBUG] nonmetric/FP/top3/U50 k=4 (eff=4), sizes: 18,24,81,128

  silhouette = 0.341230

  dunn = 0.066682

  DB = 0.992025

  CH = 92.677418

  ptbiserial = 0.495763


[DEBUG] nonmetric/FP/top3/U50 k=5 (eff=5), sizes: 14,18,24,81,114

  silhouette = 0.353859

  dunn = 0.066682

  DB = 0.895188

  CH = 105.507965

  ptbiserial = 0.553127


[DEBUG] nonmetric/FP/top3/U50 k=6 (eff=6), sizes: 14,18,24,25,56,114

  silhouette = 0.414965

  dunn = 0.072686

  DB = 0.789442

  CH = 127.070564

  ptbiserial = 

  silhouette = 0.464223

  dunn = 0.115433

  DB = 0.660872

  CH = 308.641320

  ptbiserial = 0.357838


[DEBUG] nonmetric/FP/top3/U50 k=41 (eff=41), sizes: 1,1,1,2,2,2,2,2,2,2,3,3,3,3,4,4,4,4,4,4,5,5,5,5,6,7,7,7,7,7,8,8,8,9,11,11,12,12,13,16,29

  silhouette = 0.462455

  dunn = 0.115433

  DB = 0.647375

  CH = 310.959240

  ptbiserial = 0.357529


[DEBUG] nonmetric/FP/top3/U50 k=42 (eff=42), sizes: 1,1,1,2,2,2,2,2,2,2,2,3,3,3,3,4,4,4,4,4,4,5,5,5,5,6,6,7,7,7,7,7,8,8,9,11,11,12,12,13,16,29

  silhouette = 0.463052

  dunn = 0.133638

  DB = 0.648653

  CH = 313.521906

  ptbiserial = 0.356274


[DEBUG] nonmetric/FP/top3/U50 k=43 (eff=43), sizes: 1,1,1,2,2,2,2,2,2,2,2,3,3,3,3,3,4,4,4,4,4,4,4,5,5,5,5,6,6,7,7,7,7,8,8,9,11,11,12,12,13,16,29

  silhouette = 0.465030

  dunn = 0.133638

  DB = 0.656204

  CH = 316.529112

  ptbiserial = 0.355048


[DEBUG] nonmetric/FP/top3/U50 k=44 (eff=44), sizes: 1,1,1,2,2,2,2,2,2,2,2,3,3,3,3,3,4,4,4,4,4,4,4,5,5,5,5,5,6,6,7,7,7,7,8,8,9,11,11,12,12,13,16,

initial  value 6.670112 
iter   5 value 4.787361
iter  10 value 4.425699
iter  15 value 4.391965
final  value 4.381211 
converged


  [Info] isoMDS final stress: 4.3812

[Info] nonmetric / U50 / FP / cumeig: n=251, p=5


=== [Compute] method=nonmetric, dataset=FP, mode=cumeig, U=U50 (k=2..50) ===


[DEBUG] nonmetric/FP/cumeig/U50 k=2 (eff=2), sizes: 51,200

  silhouette = 0.235306

  dunn = 0.083594

  DB = 1.672295

  CH = 48.675197

  ptbiserial = 0.336830


[DEBUG] nonmetric/FP/cumeig/U50 k=3 (eff=3), sizes: 51,58,142

  silhouette = 0.238612

  dunn = 0.088257

  DB = 1.691736

  CH = 52.163693

  ptbiserial = 0.445711


[DEBUG] nonmetric/FP/cumeig/U50 k=4 (eff=4), sizes: 18,51,58,124

  silhouette = 0.277172

  dunn = 0.088257

  DB = 1.366610

  CH = 56.496260

  ptbiserial = 0.510353


[DEBUG] nonmetric/FP/cumeig/U50 k=5 (eff=5), sizes: 18,25,26,58,124

  silhouette = 0.328791

  dunn = 0.088257

  DB = 1.171406

  CH = 62.616321

  ptbiserial = 0.536437


[DEBUG] nonmetric/FP/cumeig/U50 k=6 (eff=6), sizes: 18,21,25,26,37,124

  silhouette = 0.360396

  dunn = 0.097134

  DB = 0.973943

  CH = 73.417324

  p

  silhouette = 0.458674

  dunn = 0.216294

  DB = 0.805836

  CH = 110.487420

  ptbiserial = 0.468120


[DEBUG] nonmetric/FP/cumeig/U50 k=41 (eff=41), sizes: 1,1,2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,4,4,4,5,5,5,5,6,6,7,7,7,7,7,8,8,8,8,8,9,14,17,18,33

  silhouette = 0.456836

  dunn = 0.216294

  DB = 0.780381

  CH = 110.908780

  ptbiserial = 0.467987


[DEBUG] nonmetric/FP/cumeig/U50 k=42 (eff=42), sizes: 1,1,2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,4,4,4,5,5,5,5,5,6,6,7,7,7,7,7,8,8,8,8,9,14,17,18,33

  silhouette = 0.457627

  dunn = 0.216294

  DB = 0.768355

  CH = 111.095009

  ptbiserial = 0.466174


[DEBUG] nonmetric/FP/cumeig/U50 k=43 (eff=43), sizes: 1,1,2,2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,4,4,4,5,5,5,5,5,5,6,6,7,7,7,7,8,8,8,8,9,14,17,18,33

  silhouette = 0.454340

  dunn = 0.216294

  DB = 0.779921

  CH = 111.130396

  ptbiserial = 0.465371


[DEBUG] nonmetric/FP/cumeig/U50 k=44 (eff=44), sizes: 1,1,1,2,2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,4,4,5,5,5,5,5,5,6,6,7,7,7,7,8,8,8,8,9,14,17,18,33



In [6]:
#!/usr/bin/env Rscript

# =============================================================================
# MDS_index_profiles_OHFP_compute_v2.R
# -----------------------------------------------------------------------------
# Purpose:
#   Compute ONLY:
#     - classical MDS (cmdscale)  &  non-metric MDS (isoMDS)
#     - For each:
#         method_tag ∈ {classical, nonmetric}
#         U_tag      ∈ {U10, U25, U50}  (max k)
#         dataset    ∈ {OH, FP}
#         mode       ∈ {top3, cumeig}
#       → Ward.D2 (Euclidean) clustering for k = 2..K_max
#       → indices: silhouette, dunn, gap, ch, db, ptbiserial
#
# Inputs:
#   <base_root>/20251026_under_25clusters_program_and_result/for_checking_20251127/data/for_MDS_STEP2/
#     - preprocessed_features_OH.csv
#     - preprocessed_features_FP.csv
#
# Outputs (CSV only; NO figures here):
#   <base_root>/20251026_under_25clusters_program_and_result/for_checking_20251127/4.2_clustering/OH_FP_only/
#     {method_tag}/{U_tag}/{dataset}/csv_profiles/
#       - index_profile_{mode}_{index}_{method_tag}_{U_tag}_kmax{K}_{TS}.csv
#       - index_profiles_all_{dataset}_{mode}_{method_tag}_{U_tag}_kmax{K}_{TS}.csv
#
#   (Script 2 for plotting / summaries can reuse these CSVs.)
# =============================================================================

# ==== CONFIG (edit here) =====================================================

base_root <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No"

# Input feature files (OH/FP) for MDS
data_dir <- file.path(
  base_root,
  "20251026_under_25clusters_program_and_result",
  "for_checking_20251127",
  "data",
  "for_MDS_STEP2"
)

# Root of all outputs for this script
out_root <- file.path(
  base_root,
  "20251026_under_25clusters_program_and_result",
  "for_checking_20251127",
  "4.2_clustering",
  "OH_FP_only"
)

# Max cluster numbers: k = 2..K_max for each tag
max_k_list <- c(U100 = 100)

# MDS methods
method_tags <- c("classical", "nonmetric")

# Gap statistic bootstrap iterations
gap_B <- 20

# Input feature file names
dataset_files <- c(
  OH = "preprocessed_features_OH.csv",
  FP = "preprocessed_features_FP.csv"
)

# Dimension selection modes
dim_modes   <- c("top3", "cumeig")

# Indices to compute
index_names <- c("silhouette", "dunn", "gap", "ch", "db", "ptbiserial")

# Timestamp for this compute run
ts_str <- format(Sys.time(), "%Y%m%d_%H%M%S")

# =============================================================================
# Libraries & basic helpers
# =============================================================================

safe_lib <- function(pkg){
  if (!require(pkg, character.only = TRUE)) {
    install.packages(pkg, repos = "https://cloud.r-project.org")
    library(pkg, character.only = TRUE)
  }
}
safe_lib("readr")
safe_lib("cluster")  # silhouette, clusGap
safe_lib("fpc")      # dunn via cluster.stats
safe_lib("MASS")     # isoMDS

set.seed(42)

ensure_dir <- function(p){
  if (!dir.exists(p)) dir.create(p, recursive = TRUE, showWarnings = FALSE)
}

read_numeric_matrix <- function(path){
  df <- read.delim(path, header = TRUE, sep = ",",
                   row.names = 1, as.is = TRUE, strip.white = FALSE)
  is_num <- !vapply(df, is.character, logical(1))
  if (!any(is_num)) stop("No numeric columns: ", path)
  df[, is_num, drop = FALSE]
}

drop_zero_var_cols <- function(M){
  M <- as.matrix(M)
  keep <- apply(M, 2, function(v) sd(v, na.rm=TRUE) > 0)
  if (!any(keep)) stop("All columns have zero variance.")
  M[, keep, drop=FALSE]
}

# =============================================================================
# MDS (classical) from correlation + distance
# =============================================================================

compute_mds_and_dist <- function(numData, k_max_eig = 50){
  corData <- suppressWarnings(cor(numData, use = "pairwise.complete.obs"))
  corData[is.na(corData)] <- 0
  ddata <- dist(1 - corData)

  k <- min(k_max_eig, max(1, ncol(corData) - 1))
  mds <- cmdscale(ddata, k = k, eig = TRUE)

  eig_vals <- mds$eig
  totaleig <- sum(abs(eig_vals))
  peigen   <- if (totaleig == 0) rep(0, length(eig_vals)) else abs(eig_vals) / totaleig
  contrib <- data.frame(
    axis        = seq_along(eig_vals),
    eigenvalue  = eig_vals,
    percent     = peigen,
    cum_percent = cumsum(peigen),
    stringsAsFactors = FALSE
  )
  list(D = ddata, mds = mds, contrib = contrib)
}

select_dims <- function(contrib_df, mode, avail_dims){
  pos_idx <- which(contrib_df$percent > 0)
  if (length(pos_idx) == 0) return(min(3, max(1, avail_dims)))
  if (mode == "top3"){
    k <- min(3, length(pos_idx))
  } else {
    thr <- 0.8
    k <- which(contrib_df$cum_percent[pos_idx] >= thr)[1]
    if (is.na(k)) k <- length(pos_idx)
  }
  k <- min(k, avail_dims); k <- max(1, k)
  k
}

# =============================================================================
# Index implementations (CH / DB / point-biserial)
# =============================================================================

compute_db_index <- function(X, cl){
  if (length(unique(cl)) < 2) return(NA_real_)
  X <- as.matrix(X)
  labs <- sort(unique(cl))
  k <- length(labs)

  cent <- sapply(labs, function(L){ colMeans(X[cl == L, , drop = FALSE]) })
  cent <- t(cent)

  S <- numeric(k)
  for (i in seq_len(k)){
    idx <- which(cl == labs[i])
    if (length(idx) <= 1){
      S[i] <- 0
    } else {
      ci <- cent[i, , drop = FALSE]
      S[i] <- mean(sqrt(rowSums((X[idx, , drop = FALSE] -
                                   matrix(ci, nrow=length(idx), ncol=ncol(X), byrow = TRUE))^2)))
    }
  }

  Dc <- as.matrix(dist(cent))
  R <- matrix(NA_real_, k, k)
  for (i in seq_len(k)){
    for (j in seq_len(k)){
      if (i == j) next
      if (Dc[i, j] == 0) R[i, j] <- Inf else R[i, j] <- (S[i] + S[j]) / Dc[i, j]
    }
  }
  db <- mean(apply(R, 1, function(v) max(v[is.finite(v)], na.rm = TRUE)), na.rm = TRUE)
  if (!is.finite(db)) return(NA_real_)
  db
}

compute_ptbiserial_index <- function(D, cl){
  n <- attr(D, "Size")
  if (length(unique(cl)) < 2 || n < 2) return(NA_real_)
  s <- logical(n * (n - 1) / 2)
  idx <- 1
  for (i in 1:(n-1)){
    for (j in (i+1):n){
      s[idx] <- (cl[i] == cl[j]); idx <- idx + 1
    }
  }
  d <- as.numeric(D)
  if (length(unique(s)) < 2) return(NA_real_)
  r <- suppressWarnings(cor(d, as.numeric(s), method = "pearson", use = "complete.obs"))
  if (is.na(r)) return(NA_real_)
  -r
}

compute_ch_index <- function(X, cl){
  X <- as.matrix(X)
  n <- nrow(X)
  labs <- sort(unique(cl))
  k <- length(labs)
  if (k < 2 || n <= k) return(NA_real_)

  m_all <- colMeans(X)
  ssb <- 0
  ssw <- 0
  for (g in labs){
    idx <- which(cl == g)
    Xg  <- X[idx, , drop = FALSE]
    mg  <- colMeans(Xg)
    ssb <- ssb + nrow(Xg) * sum((mg - m_all)^2)
    ssw <- ssw + sum(rowSums((Xg - matrix(mg, nrow = nrow(Xg),
                                          ncol = ncol(Xg), byrow = TRUE))^2))
  }
  if (ssw <= 0) return(NA_real_)
  (ssb / (k - 1)) / (ssw / (n - k))
}

# =============================================================================
# Get MDS coordinates (classical / nonmetric)
# =============================================================================

get_X_for <- function(dataset_tag = c("OH","FP"),
                      mode = c("top3","cumeig"),
                      method_tag = c("classical","nonmetric")){
  dataset_tag <- match.arg(dataset_tag)
  mode        <- match.arg(mode)
  method_tag  <- match.arg(method_tag)

  csv_name <- dataset_files[[dataset_tag]]
  in_csv <- file.path(data_dir, csv_name)
  if (!file.exists(in_csv)) stop("Input file not found: ", in_csv)

  message(sprintf("\n[Info] Reading features: %s (%s)", in_csv, dataset_tag))
  numData <- read_numeric_matrix(in_csv)

  md <- compute_mds_and_dist(numData)
  D   <- md$D
  mds <- md$mds
  contrib <- md$contrib

  coords_classical <- as.matrix(mds$points)
  if (is.null(dim(coords_classical))) {
    coords_classical <- matrix(coords_classical, ncol = 1)
    colnames(coords_classical) <- "Dim1"
  }

  k_dim <- select_dims(contrib, mode, ncol(coords_classical))
  message(sprintf("[Info] %s / %s: selected dim = %d (method=%s)",
                  dataset_tag, mode, k_dim, method_tag))

  if (method_tag == "classical"){
    X <- coords_classical[, 1:k_dim, drop = FALSE]
  } else {
    init <- coords_classical[, 1:k_dim, drop = FALSE]
    iso <- try(MASS::isoMDS(D, y = init, k = k_dim, maxit = 50, trace = TRUE),
               silent = TRUE)
    if (inherits(iso, "try-error")){
      message("  [Warn] isoMDS failed → fall back to classical coords")
      X <- init
    } else {
      message(sprintf("  [Info] isoMDS final stress: %.4f", iso$stress))
      X <- iso$points
    }
  }

  X <- scale(drop_zero_var_cols(X))
  list(X = X, D = dist(X))
}

# =============================================================================
# Index profile computation over k = 2..K_max
# =============================================================================

ward_cut <- function(x, k){
  cl <- cutree(hclust(dist(x), method = "ward.D2"), k)
  list(cluster = cl)
}

compute_index_profiles <- function(
  X,
  D,
  dataset_tag,
  mode_tag,
  U_tag,
  k_max,
  out_csv_dir,
  method_tag
){
  ensure_dir(out_csv_dir)

  n <- nrow(X)
  k_max <- min(k_max, max(2, n - 1))
  ks <- 2:k_max

  message(sprintf("\n=== [Compute] method=%s, dataset=%s, mode=%s, U=%s (k=2..%d) ===",
                  method_tag, dataset_tag, mode_tag, U_tag, k_max))

  # ---- GAP (cluster::clusGap) ----
  gap_vec <- rep(NA_real_, length(ks))
  if (n >= 3) {
    set.seed(42)
    gap_obj <- try(
      cluster::clusGap(X, FUNcluster = ward_cut,
                       K.max = k_max, B = gap_B, verbose = FALSE),
      silent = TRUE
    )
    if (!inherits(gap_obj, "try-error")) {
      gap_tab <- as.data.frame(gap_obj$Tab)
      gap_tab$k <- seq_len(nrow(gap_tab))
      gap_tab <- gap_tab[gap_tab$k %in% ks, c("k","gap")]
      for (j in seq_len(nrow(gap_tab))) {
        pos <- which(ks == gap_tab$k[j])
        if (length(pos) == 1) gap_vec[pos] <- gap_tab$gap[j]
      }
    } else {
      warning("clusGap failed for ", dataset_tag, "/", mode_tag, "/", U_tag,
              " (method=", method_tag, "): ",
              conditionMessage(attr(gap_obj, "condition")))
    }
  }

  # ---- other indices over k ----
  sil  <- dunn <- db <- ch <- ptb <- rep(NA_real_, length(ks))

  hc <- hclust(D, method = "ward.D2")

  for (i in seq_along(ks)){
    k <- ks[i]
    cl_raw <- cutree(hc, k)
    cl <- as.integer(factor(cl_raw))
    k_eff <- length(unique(cl))
    sizes <- sort(table(cl))

    message(sprintf("\n[DEBUG] %s/%s/%s/%s k=%d (eff=%d), sizes: %s",
                    method_tag, dataset_tag, mode_tag, U_tag,
                    k, k_eff, paste(sizes, collapse = ",")))

    if (k_eff < 2){
      message("  -> k_eff < 2, all indices set to NA")
      next
    }

    # silhouette
    sil[i] <- tryCatch({
      sil_obj <- cluster::silhouette(cl, D)
      val <- mean(sil_obj[,3], na.rm = TRUE)
      message(sprintf("  silhouette = %s",
                      ifelse(is.na(val), "NA", sprintf("%.6f", val))))
      val
    }, error = function(e){
      message(sprintf("  [ERROR silhouette] %s", conditionMessage(e)))
      NA_real_
    })

    # dunn (via fpc::cluster.stats)
    dstat <- try(fpc::cluster.stats(D, cl), silent = TRUE)
    if (!inherits(dstat, "try-error") && !is.null(dstat$dunn)){
      dunn[i] <- dstat$dunn
      message(sprintf("  dunn = %s",
                      ifelse(is.na(dunn[i]), "NA", sprintf("%.6f", dunn[i]))))
    } else if (inherits(dstat, "try-error")){
      message(sprintf("  [ERROR dunn] %s",
                      conditionMessage(attr(dstat, "condition"))))
    }

    # DB
    db[i] <- tryCatch({
      val <- compute_db_index(X, cl)
      message(sprintf("  DB = %s",
                      ifelse(is.na(val), "NA", sprintf("%.6f", val))))
      val
    }, error = function(e){
      message(sprintf("  [ERROR DB] %s", conditionMessage(e)))
      NA_real_
    })

    # CH
    ch[i] <- tryCatch({
      val <- compute_ch_index(X, cl)
      message(sprintf("  CH = %s",
                      ifelse(is.na(val), "NA", sprintf("%.6f", val))))
      val
    }, error = function(e){
      message(sprintf("  [ERROR CH] %s", conditionMessage(e)))
      NA_real_
    })

    # ptbiserial
    ptb[i] <- tryCatch({
      val <- compute_ptbiserial_index(D, cl)
      message(sprintf("  ptbiserial = %s",
                      ifelse(is.na(val), "NA", sprintf("%.6f", val))))
      val
    }, error = function(e){
      message(sprintf("  [ERROR ptbiserial] %s", conditionMessage(e)))
      NA_real_
    })
  }

  # ---- save per-index CSVs ----
  profiles <- list(
    silhouette = data.frame(k = ks, score = sil),
    dunn       = data.frame(k = ks, score = dunn),
    gap        = data.frame(k = ks, score = gap_vec),
    ch         = data.frame(k = ks, score = ch),
    db         = data.frame(k = ks, score = db),
    ptbiserial = data.frame(k = ks, score = ptb)
  )

  for (ix in names(profiles)){
    df <- profiles[[ix]]
    out_f <- file.path(
      out_csv_dir,
      sprintf("index_profile_%s_%s_%s_%s_kmax%d_%s.csv",
              mode_tag, ix, method_tag, U_tag, k_max, ts_str)
    )
    readr::write_csv(df, out_f)
    message("[Saved] ", out_f)
  }

  # ---- save combined all-k CSV ----
  all_k_df <- data.frame(
    k          = ks,
    silhouette = sil,
    dunn       = dunn,
    gap        = gap_vec,
    ch         = ch,
    db         = db,
    ptbiserial = ptb,
    stringsAsFactors = FALSE
  )
  out_all <- file.path(
    out_csv_dir,
    sprintf("index_profiles_all_%s_%s_%s_%s_kmax%d_%s.csv",
            dataset_tag, mode_tag, method_tag, U_tag, k_max, ts_str)
  )
  readr::write_csv(all_k_df, out_all)
  message("[Saved all-k indices] ", out_all)

  invisible(NULL)
}

# =============================================================================
# MAIN
# =============================================================================

ensure_dir(out_root)

for (method_tag in method_tags){
  message(sprintf("\n================ METHOD = %s ================\n", method_tag))
  method_root <- file.path(out_root, method_tag)
  ensure_dir(method_root)

  for (U_tag in names(max_k_list)){
    k_max <- max_k_list[[U_tag]]
    message(sprintf("\n######## U-tag = %s (k_max = %d) ########", U_tag, k_max))

    out_U_root <- file.path(method_root, U_tag)
    ensure_dir(out_U_root)

    for (dataset_tag in names(dataset_files)){
      out_ds_root <- file.path(out_U_root, dataset_tag)
      out_ds_csv  <- file.path(out_ds_root, "csv_profiles")
      ensure_dir(out_ds_root); ensure_dir(out_ds_csv)

      for (mode_tag in dim_modes){

        mdx <- get_X_for(dataset_tag, mode_tag, method_tag)
        X <- as.matrix(mdx$X)
        D <- mdx$D

        message(sprintf("[Info] %s / %s / %s / %s: n=%d, p=%d",
                        method_tag, U_tag, dataset_tag, mode_tag,
                        nrow(X), ncol(X)))

        compute_index_profiles(
          X          = X,
          D          = D,
          dataset_tag = dataset_tag,
          mode_tag    = mode_tag,
          U_tag       = U_tag,
          k_max       = k_max,
          out_csv_dir = out_ds_csv,
          method_tag  = method_tag
        )
      }
    }
  }
}

message("\n✅ Compute finished. CSV profiles are under: ", out_root)



================ METHOD = classical ================



######## U-tag = U100 (k_max = 100) ########


[Info] Reading features: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/for_MDS_STEP2/preprocessed_features_OH.csv (OH)

[Info] OH / top3: selected dim = 3 (method=classical)

[Info] classical / U100 / OH / top3: n=394, p=3


=== [Compute] method=classical, dataset=OH, mode=top3, U=U100 (k=2..100) ===


[DEBUG] classical/OH/top3/U100 k=2 (eff=2), sizes: 20,374

  silhouette = 0.774703

  dunn = 0.109484

  DB = 1.123348

  CH = 132.625585

  ptbiserial = 0.719927


[DEBUG] classical/OH/top3/U100 k=3 (eff=3), sizes: 7,13,374

  silhouette = 0.779770

  dunn = 0.109484

  DB = 0.446201

  CH = 185.494444

  ptbiserial = 0.729326


[DEBUG] classical/OH/top3/U100 k=4 (eff=4), sizes: 7,9,13,365

  silhouette = 0.763396

  dunn = 0.063974

  DB = 0.424894

  CH = 226.244865

  ptbiserial = 0.814971


[DEBU

  silhouette = 0.390606

  dunn = 0.017965

  DB = 0.486241

  CH = 859.579421

  ptbiserial = 0.309925


[DEBUG] classical/OH/top3/U100 k=40 (eff=40), sizes: 1,1,1,1,1,1,1,1,1,1,1,1,1,2,2,2,2,2,2,3,3,3,3,3,4,4,5,5,6,6,6,8,8,9,16,17,32,49,60,119

  silhouette = 0.389013

  dunn = 0.025373

  DB = 0.481472

  CH = 887.687713

  ptbiserial = 0.309929


[DEBUG] classical/OH/top3/U100 k=41 (eff=41), sizes: 1,1,1,1,1,1,1,1,1,1,1,1,1,1,2,2,2,2,2,2,3,3,3,3,3,4,4,5,5,6,6,6,7,8,9,16,17,32,49,60,119

  silhouette = 0.390626

  dunn = 0.025373

  DB = 0.471014

  CH = 914.244166

  ptbiserial = 0.309918


[DEBUG] classical/OH/top3/U100 k=42 (eff=42), sizes: 1,1,1,1,1,1,1,1,1,1,1,1,1,1,2,2,2,2,2,2,2,3,3,3,3,3,4,4,5,5,6,6,6,6,7,9,16,17,32,49,60,119

  silhouette = 0.390639

  dunn = 0.025373

  DB = 0.469600

  CH = 938.979289

  ptbiserial = 0.309848


[DEBUG] classical/OH/top3/U100 k=43 (eff=43), sizes: 1,1,1,1,1,1,1,1,1,1,1,1,1,1,2,2,2,2,2,2,2,3,3,3,3,3,4,4,5,5,6,6,6,6,7,9,16,17,32,44,49,60,75



  silhouette = 0.360127

  dunn = 0.031877

  DB = 0.397635

  CH = 1656.554199

  ptbiserial = 0.203540


[DEBUG] classical/OH/top3/U100 k=70 (eff=70), sizes: 1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,2,2,2,2,2,2,2,2,2,2,2,2,3,3,3,3,3,3,3,4,5,5,6,6,8,9,12,12,13,15,19,31,32,38,44,56

  silhouette = 0.360986

  dunn = 0.031877

  DB = 0.418899

  CH = 1697.423996

  ptbiserial = 0.203371


[DEBUG] classical/OH/top3/U100 k=71 (eff=71), sizes: 1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,2,2,2,2,2,2,2,2,2,2,2,2,3,3,3,3,3,3,3,4,5,5,5,6,6,8,9,12,12,13,15,19,31,32,38,39,56

  silhouette = 0.368721

  dunn = 0.031877

  DB = 0.418293

  CH = 1740.448577

  ptbiserial = 0.199485


[DEBUG] classical/OH/top3/U100 k=72 (eff=72), sizes: 1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,2,2,2,2,2,2,2,2,2,2,2,2,2,3,3,3,3,3,3,4,5,5,5,6,6,8,9,12,12,13,15,19,31,32,38,39,56

  silhouette = 0.369491

  dunn = 0.033941

  DB = 0.400638



  silhouette = 0.346656

  dunn = 0.052725

  DB = 0.344502

  CH = 2980.018095

  ptbiserial = 0.162369


[DEBUG] classical/OH/top3/U100 k=95 (eff=95), sizes: 1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,2,2,2,2,2,2,2,2,2,2,2,2,3,3,3,3,3,4,4,4,4,5,5,5,6,7,8,9,10,12,12,13,13,15,15,16,17,19,23,31,44

  silhouette = 0.348254

  dunn = 0.052725

  DB = 0.357920

  CH = 3034.307499

  ptbiserial = 0.156274


[DEBUG] classical/OH/top3/U100 k=96 (eff=96), sizes: 1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,2,2,2,2,2,2,2,2,2,2,2,2,3,3,3,3,3,4,4,4,4,5,5,5,6,7,8,9,10,12,12,13,13,15,15,16,17,19,20,23,24,31

  silhouette = 0.346739

  dunn = 0.040821

  DB = 0.360531

  CH = 3092.626343

  ptbiserial = 0.141999


[DEBUG] classical/OH/top3/U100 k=97 (eff=97), sizes: 1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,2

  dunn = 0.168493

  DB = 1.211877

  CH = 9.652775

  ptbiserial = 0.477590


[DEBUG] classical/OH/cumeig/U100 k=22 (eff=22), sizes: 2,2,2,2,2,2,2,3,3,3,4,4,4,5,5,6,7,10,15,22,23,266

  silhouette = 0.250077

  dunn = 0.168493

  DB = 1.406150

  CH = 9.812648

  ptbiserial = 0.528687


[DEBUG] classical/OH/cumeig/U100 k=23 (eff=23), sizes: 2,2,2,2,2,2,2,3,3,3,4,4,4,5,5,6,7,8,10,14,15,23,266

  silhouette = 0.256694

  dunn = 0.168493

  DB = 1.407724

  CH = 9.978682

  ptbiserial = 0.530418


[DEBUG] classical/OH/cumeig/U100 k=24 (eff=24), sizes: 2,2,2,2,2,2,2,3,3,3,4,4,4,5,5,6,7,7,8,8,10,14,23,266

  silhouette = 0.233340

  dunn = 0.168493

  DB = 1.384982

  CH = 10.155698

  ptbiserial = 0.530964


[DEBUG] classical/OH/cumeig/U100 k=25 (eff=25), sizes: 2,2,2,2,2,2,2,2,3,3,3,3,4,4,4,5,6,7,7,8,8,10,14,23,266

  silhouette = 0.241649

  dunn = 0.168493

  DB = 1.321879

  CH = 10.343109

  ptbiserial = 0.531226


[DEBUG] classical/OH/cumeig/U100 k=26 (eff=26), sizes: 2,2,2,2,2,2,2,

...

initial  value 6.670112 
iter   5 value 4.787361
iter  10 value 4.425699
iter  15 value 4.391965
final  value 4.381211 
converged


  [Info] isoMDS final stress: 4.3812

[Info] nonmetric / U100 / FP / cumeig: n=251, p=5


=== [Compute] method=nonmetric, dataset=FP, mode=cumeig, U=U100 (k=2..100) ===


[DEBUG] nonmetric/FP/cumeig/U100 k=2 (eff=2), sizes: 51,200

  silhouette = 0.235306

  dunn = 0.083594

  DB = 1.672295

  CH = 48.675197

  ptbiserial = 0.336830


[DEBUG] nonmetric/FP/cumeig/U100 k=3 (eff=3), sizes: 51,58,142

  silhouette = 0.238612

  dunn = 0.088257

  DB = 1.691736

  CH = 52.163693

  ptbiserial = 0.445711


[DEBUG] nonmetric/FP/cumeig/U100 k=4 (eff=4), sizes: 18,51,58,124

  silhouette = 0.277172

  dunn = 0.088257

  DB = 1.366610

  CH = 56.496260

  ptbiserial = 0.510353


[DEBUG] nonmetric/FP/cumeig/U100 k=5 (eff=5), sizes: 18,25,26,58,124

  silhouette = 0.328791

  dunn = 0.088257

  DB = 1.171406

  CH = 62.616321

  ptbiserial = 0.536437


[DEBUG] nonmetric/FP/cumeig/U100 k=6 (eff=6), sizes: 18,21,25,26,37,124

  silhouette = 0.360396

  dunn = 0.097134

  DB = 0.973943

  CH = 73.417

  silhouette = 0.458674

  dunn = 0.216294

  DB = 0.805836

  CH = 110.487420

  ptbiserial = 0.468120


[DEBUG] nonmetric/FP/cumeig/U100 k=41 (eff=41), sizes: 1,1,2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,4,4,4,5,5,5,5,6,6,7,7,7,7,7,8,8,8,8,8,9,14,17,18,33

  silhouette = 0.456836

  dunn = 0.216294

  DB = 0.780381

  CH = 110.908780

  ptbiserial = 0.467987


[DEBUG] nonmetric/FP/cumeig/U100 k=42 (eff=42), sizes: 1,1,2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,4,4,4,5,5,5,5,5,6,6,7,7,7,7,7,8,8,8,8,9,14,17,18,33

  silhouette = 0.457627

  dunn = 0.216294

  DB = 0.768355

  CH = 111.095009

  ptbiserial = 0.466174


[DEBUG] nonmetric/FP/cumeig/U100 k=43 (eff=43), sizes: 1,1,2,2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,4,4,4,5,5,5,5,5,5,6,6,7,7,7,7,8,8,8,8,9,14,17,18,33

  silhouette = 0.454340

  dunn = 0.216294

  DB = 0.779921

  CH = 111.130396

  ptbiserial = 0.465371


[DEBUG] nonmetric/FP/cumeig/U100 k=44 (eff=44), sizes: 1,1,1,2,2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,4,4,5,5,5,5,5,5,6,6,7,7,7,7,8,8,8,8,9,14,17,18,

  silhouette = 0.413654

  dunn = 0.217302

  DB = 0.694096

  CH = 124.683339

  ptbiserial = 0.372682


[DEBUG] nonmetric/FP/cumeig/U100 k=71 (eff=71), sizes: 1,1,1,1,1,1,1,1,1,1,1,1,1,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,4,4,4,4,4,4,5,5,5,5,5,5,6,6,6,6,7,8,9,10,11,12,18

  silhouette = 0.405941

  dunn = 0.169601

  DB = 0.703919

  CH = 125.583034

  ptbiserial = 0.346091


[DEBUG] nonmetric/FP/cumeig/U100 k=72 (eff=72), sizes: 1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,4,4,4,4,4,4,5,5,5,5,5,5,6,6,6,6,7,8,9,10,11,12,18

  silhouette = 0.406891

  dunn = 0.169601

  DB = 0.693860

  CH = 126.586395

  ptbiserial = 0.345812


[DEBUG] nonmetric/FP/cumeig/U100 k=73 (eff=73), sizes: 1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,4,4,4,4,4,4,4,5,5,5,5,5,5,6,6,6,6,8,9,10,11,12,18

  silhouette = 0.411119

  dunn = 0.169601

  DB = 0.689306

  CH = 12

  silhouette = 0.413573

  dunn = 0.246589

  DB = 0.492158

  CH = 164.638927

  ptbiserial = 0.321870


[DEBUG] nonmetric/FP/cumeig/U100 k=96 (eff=96), sizes: 1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,3,4,4,4,4,4,4,5,5,5,5,6,6,6,6,8,9,10,12,17

  silhouette = 0.411924

  dunn = 0.246589

  DB = 0.490749

  CH = 166.388314

  ptbiserial = 0.321453


[DEBUG] nonmetric/FP/cumeig/U100 k=97 (eff=97), sizes: 1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,3,4,4,4,4,4,4,5,5,5,5,6,6,6,6,8,9,10,12,17

  silhouette = 0.411243

  dunn = 0.246589

  DB = 0.481056

  CH = 168.287222

  ptbiserial = 0.321256


[DEBUG] nonmetric/FP/cumeig/U100 k=98 (eff=98), sizes: 1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2